### Test for the whole workflow

## Setup
Set up file system for the datset(using Google Drive), dagshub and MLflow

In [1]:
# Install dependencies
%%capture
%pip install -q dagshub[jupyter]
!pip install mlflow
!pip install rasterio
!pip install natsort

# Suppress warnings
import logging
logging.getLogger("rasterio").setLevel(logging.ERROR)

In [2]:
# Mount google drive
from google.colab import drive
drive.mount('/content/drive')

# Set up dagshub
!dagshub login

import dagshub
TOKEN = dagshub.auth.get_token()

Mounted at /content/drive
                                ❗❗❗ AUTHORIZATION REQUIRED ❗❗❗                                


Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=59033a8c-ba3c-478a-ae41-f2b2d506a341&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=e69bdb845d7f4edcfc7fe2b1ac12f582f74e5bfb5130b8491cfb93f855fc6f05


⠸ Waiting for authorization
✅ OAuth token added


Accessing as chengzwk

In [3]:
# Read data
import os
import rasterio
import numpy as np
from natsort import natsorted

data_dir = "/content/drive/MyDrive/Omdena/urban-green-frankfurt/MULC"
image_dir = 'VBWVA_8R'
image_files = [f for f in os.listdir(os.path.join(data_dir, image_dir)) if f.endswith('1_GeoTIFF.tif')]
image_files = natsorted(image_files)
image_path = os.path.join(data_dir, image_dir, image_files[0])
with (rasterio.open(image_path) as img):
    image_array = img.read()
image_array = np.transpose(image_array, [1, 2, 0])  # move the axis for bands to the third axis
image_array[np.isnan(image_array)] = 0              # replace nan with 0
image_array = image_array[:, :, (1, 2, 3)]

# Create a directory for the results in Google Drive
results_dir = '/content/drive/My Drive/dummy_results'
os.makedirs(results_dir, exist_ok=True)

# Make a plot
import matplotlib.pyplot as plt
plt.imshow(image_array[:, :, 1])
plt.savefig(os.path.join(results_dir, 'image.png'))
plt.close()

# Create a dummy numpy array of ~1MB and save it to the results directory
np.save(os.path.join(results_dir, 'image_array.npy'), image_array)

print(f"results saved to: {results_dir}")

results saved to: /content/drive/My Drive/dummy_results


In [ ]:
from dagshub.notebook import save_notebook

repo = "chengzwk/omdena-frankfurt-ugs-unet"
branch = "overfit-experiments"
notebook_path = "unet-from-scratch.ipynb"
commit_message = "Turn off output for installing dependencies"

save_notebook(
    repo=repo,
    branch=branch,
    path=notebook_path,
    commit_message=commit_message,
    versioning="git"
    )